# Closed-loop index on published Atari agents

Applies the closed-loop index (CLI) from `Mario_RL_Audit.ipynb` to PPO agents trained and published by the Stable-Baselines3 RL Zoo (`sb3/ppo-<Game>NoFrameskip-v4` on Hugging Face). Nothing is trained.

For every game and episode *k*:

1. **clean**: the agent plays one full game (all lives); its button presses are recorded as a *tape*.
2. **tape_clean**: the tape is replayed blind. The emulator is deterministic, so this must reproduce the clean game exactly; games that fail this check are excluded.
3. **sticky / tape_sticky**: the agent, and its tape, play under sticky actions (each decision repeats the previous one with p = 0.25), with the *same* sticky draws at every step.
4. **noop / tape_noop**: the same with 0–30 extra idle decisions at the start.

**Kept** = (score − random) ÷ (clean score − random). **CLI** = kept by the agent − kept by its tapes. CLI ≈ 0: the agent does no better than a recording of itself; CLI > 0: it uses the screen to recover.

Run headless with `NOTEBOOK=Atari_CLI_Audit.ipynb ./run_experiments.sh`. Outputs go to `results_atari/`; finished episodes are cached in `results_atari/raw/`.

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys
need = {"stable_baselines3": "stable-baselines3", "ale_py": "ale-py", "huggingface_hub": "huggingface_hub",
        "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib", "cv2": "opencv-python-headless"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

In [ ]:
import json, multiprocessing as mp, os, platform, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
from gymnasium import spaces
import ale_py
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.atari_wrappers import AtariWrapper, EpisodicLifeEnv
from scipy import stats
from IPython.display import display

gym.register_envs(ale_py)
warnings.filterwarnings("ignore", category=UserWarning)
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__,
            "gymnasium": gym.__version__, "ale_py": ale_py.__version__, "stable_baselines3": sb3.__version__}
print(" | ".join(f"{k} {v}" for k, v in VERSIONS.items()), "| CPU cores:", os.cpu_count())

In [ ]:
# Parameters: papermill overrides these, e.g. -y 'GAMES: [Pong, Breakout]' -p EPISODES 5
GAMES = ["Pong", "Breakout", "SpaceInvaders", "Qbert", "Seaquest", "BeamRider", "Enduro", "MsPacman", "Asteroids", "RoadRunner"]
HF_ORG = "sb3"                      # repo id: <HF_ORG>/ppo-<Game>NoFrameskip-v4
MODEL_DIR = "atari_models"          # downloaded agents; a game whose .zip is already here is not downloaded again
EPISODES = 10                       # episodes per game and condition (also for the random reference)
MAX_STEPS = 27_000                  # agent decisions per game (= 108,000 frames, the standard cap)
STICKY_P = 0.25
NOOP_MAX = 30
EP_SEED = 10_000
MIN_TAPE_LOSS = 0.2                 # a perturbation is informative for a game only if it costs the blind tape ≥ 20% of its progress
N_WORKERS = None                    # None: CPU cores − 1 (at most 12)
REEVALUATE = False

In [ ]:
GAMES = list(dict.fromkeys(GAMES))
ROOT = Path.cwd()
MODELS = ROOT / MODEL_DIR
RESULTS = ROOT / "results_atari"
DIRS = {k: RESULTS / k for k in ["figures", "tables", "raw"]}
for d in [MODELS, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)
N_WORKERS = N_WORKERS or max(1, min(12, (os.cpu_count() or 2) - 1))
CONFIG = {k: globals()[k] for k in ["GAMES", "HF_ORG", "EPISODES", "MAX_STEPS", "STICKY_P", "NOOP_MAX", "EP_SEED", "MIN_TAPE_LOSS", "N_WORKERS"]}
(RESULTS / "config.json").write_text(json.dumps({"config": CONFIG, "environment": VERSIONS}, indent=2))
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False})
BLUE, GREY, ORANGE = "#2a78d6", "#9a9a95", "#eb6834"

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(DIRS["figures"] / f"{name}.{ext}", bbox_inches="tight")
    print(f"saved → {RESULTS.name}/figures/{name}.{{png,pdf}}")

def _fmt(v, f):
    if v is None or (isinstance(v, (float, np.floating)) and np.isnan(v)):
        return "–"
    return format(v, f) if isinstance(v, (float, np.floating)) else str(v)

def save_table(df, name, caption, floatfmt=",.2f"):
    df.to_csv(DIRS["tables"] / f"{name}.csv", index=False)
    head = "| " + " | ".join(map(str, df.columns)) + " |"
    rule = "|" + "|".join([":---"] + ["---:"] * (len(df.columns) - 1)) + "|"
    rows = ["| " + " | ".join(_fmt(v, floatfmt) for v in r) + " |" for r in df.itertuples(index=False)]
    (DIRS["tables"] / f"{name}.md").write_text(f"### {caption}\n\n" + "\n".join([head, rule, *rows]) + "\n")
    print(f"saved → {RESULTS.name}/tables/{name}.{{csv,md}}")
    display(df)

def boot_ci(x, n=10_000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2:
        return np.nan, np.nan
    m = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return tuple(np.percentile(m, [2.5, 97.5]))

def holm(p):
    p = np.asarray(p, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    order = np.argsort(p[ok]); m = ok.sum()
    out[np.flatnonzero(ok)[order]] = np.maximum.accumulate((m - np.arange(m)) * p[ok][order]).clip(max=1)
    return out

## 2. Published agents

In [ ]:
def env_id(game):
    return f"{game}NoFrameskip-v4"

def model_path(game):
    return MODELS / f"ppo-{env_id(game)}" / f"ppo-{env_id(game)}.zip"

for game in GAMES:
    p = model_path(game)
    if p.exists():
        continue
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(repo_id=f"{HF_ORG}/ppo-{env_id(game)}", filename=p.name, local_dir=p.parent)
        print(f"downloaded {HF_ORG}/ppo-{env_id(game)}")
    except Exception as e:
        print(f"{game}: download failed ({type(e).__name__}: {e}); skipped")
GAMES = [g for g in GAMES if model_path(g).exists()]
print("agents available:", GAMES)

def make_env(game):
    """The RL-Zoo training wrappers (no-op reset, frame skip 4 with max-pooling, life loss = episode end, FIRE on reset,
    84×84 grayscale) without reward clipping, so the real game score is measured."""
    return AtariWrapper(gym.make(env_id(game)), clip_reward=False)

def load_agent(game):
    n = gym.make(env_id(game)).action_space.n
    custom = {"learning_rate": 0.0, "lr_schedule": lambda _: 0.0, "clip_range": lambda _: 0.0,
              "observation_space": spaces.Box(0, 255, (4, 84, 84), np.uint8), "action_space": spaces.Discrete(n)}
    model = PPO.load(model_path(game), device="cpu", custom_objects=custom)
    model.policy.set_training_mode(False)
    return model.policy

rows = []
for game in GAMES:
    pol = load_agent(game); e = make_env(game)
    rows.append({"Game": game, "Actions": e.action_space.n, "Lives-based episodes": "EpisodicLifeEnv" in str(e),
                 "FIRE on reset": "FireResetEnv" in str(e), "Policy params": sum(p.numel() for p in pol.parameters())})
    e.close()
save_table(pd.DataFrame(rows), "agents", "Published PPO agents audited (RL Zoo, Hugging Face)", ",.0f")

## 3. Playing a full game, blind replays and perturbations

One *game* is all lives. As in training, a lost life ends the wrapper's episode and the next one starts with a cleared frame stack (and FIRE where the game needs it); the emulator itself is only reset when the game is over. Actions are sampled from the policy, as in RL-Zoo Atari evaluation. Each episode *k* uses three independent seeded streams: the emulator's reset (including its random 1–30 no-ops), the policy's action sampling, and the perturbation draws. The perturbation stream is shared by the agent and its tape, so both face the same sticky steps.

In [ ]:
@torch.no_grad()
def action_probs(policy, stack):
    t, _ = policy.obs_to_tensor(stack[None])
    return policy.get_distribution(t).distribution.probs[0].numpy().astype(np.float64)

def hard_reset(game_env, seed):
    """Always start a brand-new game. EpisodicLifeEnv only resets the emulator after a real game over, so a game that
    ended any other way (end of a tape, step cap) would otherwise carry over into the next condition."""
    e = game_env
    while e is not None:
        if isinstance(e, EpisodicLifeEnv):
            e.was_real_done = True
        e = getattr(e, "env", None)
    return game_env.reset(seed=seed)

def play(game_env, seed, policy=None, tape=None, sticky=False, extra_noop=0, n_actions=None, random_play=False):
    """One full game. A tape that runs out before the game is over continues blind with NOOP (still open-loop) until
    game over, so a replay is never cut short in the agent's favour."""
    rng_act = np.random.default_rng([seed, 0]); rng_pert = np.random.default_rng([seed, 1])
    obs, _ = hard_reset(game_env, seed)
    stack = np.zeros((84, 84, 4), np.uint8); stack[..., -1] = obs[..., 0]
    score, actions, ents, prev, k, ended, tape_ran_out = 0.0, [], [], 0, 0, "max_steps", False
    for t in range(MAX_STEPS):
        if t < extra_noop:
            a = 0
        elif tape is not None:
            if k >= len(tape):
                a = 0; tape_ran_out = True
            else:
                a = tape[k]; k += 1
        elif random_play:
            a = int(rng_act.integers(n_actions))
        else:
            p = action_probs(policy, stack)
            ents.append(float(-(p[p > 0] * np.log(p[p > 0])).sum() / np.log(len(p))))
            a = int(rng_act.choice(len(p), p=p / p.sum()))
        if sticky:
            u = rng_pert.random()                       # drawn every step, so agent and tape share the draws
            if t > 0 and u < STICKY_P:
                a = prev
        prev = a; actions.append(a)
        obs, r, term, trunc, _ = game_env.step(a)
        score += float(r)
        if term or trunc:
            if trunc or game_env.unwrapped.ale.game_over():
                ended = "game_over" if not trunc else "truncated"; break
            obs, _ = game_env.reset()                   # life lost: continue the same game, as during training
            stack = np.zeros((84, 84, 4), np.uint8); stack[..., -1] = obs[..., 0]
        else:
            stack[..., :-1] = stack[..., 1:]; stack[..., -1] = obs[..., 0]
    return {"score": score, "steps": t + 1, "ended": ended, "tape_ran_out": tape_ran_out,
            "entropy": float(np.mean(ents)) if ents else np.nan}, actions

def job(args):
    """One game, one episode index k: every condition (or the random reference). Cached in results_atari/raw/."""
    game, k, kind = args
    out = DIRS["raw"] / f"{game}__{kind}__{k}.csv"
    if out.exists() and not REEVALUATE:
        return pd.read_csv(out)
    torch.set_num_threads(1)
    seed = EP_SEED + k
    env = make_env(game); rows = []
    if kind == "random":
        r, _ = play(env, seed, random_play=True, n_actions=env.action_space.n)
        rows.append({"condition": "random", **r})
    else:
        pol = load_agent(game)
        n_noop = int(np.random.default_rng([seed, 2]).integers(0, NOOP_MAX + 1))
        r, tape = play(env, seed, policy=pol); rows.append({"condition": "clean", **r})
        r, _ = play(env, seed, tape=tape); rows.append({"condition": "tape_clean", **r})
        r, _ = play(env, seed, policy=pol, sticky=True); rows.append({"condition": "sticky", **r})
        r, _ = play(env, seed, tape=tape, sticky=True); rows.append({"condition": "tape_sticky", **r})
        r, _ = play(env, seed, policy=pol, extra_noop=n_noop); rows.append({"condition": "noop", **r})
        r, _ = play(env, seed, tape=tape, extra_noop=n_noop); rows.append({"condition": "tape_noop", **r})
    env.close()
    df = pd.DataFrame(rows).assign(game=game, episode=k)
    df.to_csv(out.with_suffix(".tmp"), index=False); out.with_suffix(".tmp").rename(out)
    return df

def run_pool(fn, jobs, label):
    t0, res = time.time(), []
    pool = mp.get_context("fork").Pool(min(N_WORKERS, len(jobs)), maxtasksperchild=8)
    try:
        for i, r in enumerate(pool.imap_unordered(fn, jobs), 1):
            res.append(r); el = time.time() - t0
            if i == len(jobs) or i % max(1, len(jobs) // 20) == 0:
                print(f"[{label}] {i}/{len(jobs)} | {el / 60:.1f} min | ~{el / i * (len(jobs) - i) / 60:.1f} min left", flush=True)
        pool.close()
    except BaseException:
        pool.terminate(); print(f"[{label}] interrupted: finished jobs are saved; re-run to continue"); raise
    finally:
        pool.join()
    return res

# quick determinism smoke test on the first game before the full run
e = make_env(GAMES[0]); pol = load_agent(GAMES[0])
a, tape = play(e, EP_SEED, policy=pol); b, _ = play(e, EP_SEED, tape=tape); e.close()
print(f"{GAMES[0]}: clean score {a['score']:.0f} in {a['steps']} decisions; blind replay {b['score']:.0f} in {b['steps']} "
      f"→ {'reproduced exactly' if (a['score'], a['steps']) == (b['score'], b['steps']) else 'NOT reproduced'}")

## 4. Run

In [ ]:
jobs = [(g, k, kind) for kind in ["agent", "random"] for g in GAMES for k in range(EPISODES)]
jobs.sort(key=lambda j: j[2] != "agent")
cached = sum((DIRS["raw"] / f"{g}__{kind}__{k}.csv").exists() for g, k, kind in jobs) if not REEVALUATE else 0
print(f"{len(jobs)} jobs ({len(GAMES)} games × {EPISODES} episodes × agent/random), {cached} already saved")
EP = pd.concat(run_pool(job, jobs, "atari"), ignore_index=True)
EP.to_csv(DIRS["raw"] / "episodes.csv", index=False)
print(f"saved → {RESULTS.name}/raw/episodes.csv ({len(EP):,} games played)")

## 5. Determinism check: does every blind replay reproduce its clean game?

In [ ]:
piv = EP.pivot_table(index=["game", "episode"], columns="condition", values=["score", "steps"])
det = pd.DataFrame({"same_score": piv["score"]["clean"] == piv["score"]["tape_clean"],
                    "same_length": piv["steps"]["clean"] == piv["steps"]["tape_clean"]}).reset_index()
det["exact"] = det.same_score & det.same_length
rows = [{"Game": g, "Episodes": len(d), "Reproduced exactly": f"{d.exact.mean():.0%}"} for g, d in det.groupby("game", sort=False)]
save_table(pd.DataFrame(rows), "determinism", "Blind replay of each clean game's button tape (must be 100% for the CLI to be valid)")
VALID = [g for g, d in det.groupby("game") if d.exact.all()]
print("games with exact replays:", VALID, "| excluded:", sorted(set(GAMES) - set(VALID)))

## 6. Closed-loop index per game

Per episode *k*: kept = (score − random mean) ÷ (clean mean − random mean), for the agent and for its own tape under the same perturbation draws. CLI = agent kept − tape kept, averaged over episodes; 95% bootstrap CI over episodes; Wilcoxon signed-rank test of agent vs. tape scores, Holm-corrected over games. Games where the agent is not clearly better than random are not scored. A perturbation is only **informative** for a game if it costs the blind tape at least 20% of its progress (tape kept ≤ 0.8): if even a recording is unharmed, the test cannot tell a reacting agent from a memorising one.

In [ ]:
RND = EP[EP.condition == "random"].groupby("game").score.mean()
rows, per_ep = [], []
for g in VALID:
    d = piv.loc[g]["score"]; clean_mean = d["clean"].mean(); rnd = RND[g]; denom = clean_mean - rnd
    if denom <= max(1e-9, 0.05 * abs(rnd)):
        print(f"{g}: agent not clearly better than random (clean {clean_mean:.1f} vs random {rnd:.1f}); not scored"); continue
    row = {"Game": g, "Random": rnd, "Clean": clean_mean,
           "Entropy": EP[(EP.game == g) & (EP.condition == "clean")].entropy.mean()}
    for c in ["sticky", "noop"]:
        ka, kt = (d[c] - rnd) / denom, (d[f"tape_{c}"] - rnd) / denom
        cli = ka - kt; lo, hi = boot_ci(cli)
        try:
            p = stats.wilcoxon(d[c], d[f"tape_{c}"]).pvalue if not np.allclose(d[c], d[f"tape_{c}"]) else 1.0
        except ValueError:
            p = np.nan
        row.update({f"{c}: agent kept": ka.mean(), f"{c}: tape kept": kt.mean(), f"{c}: CLI": cli.mean(),
                    f"{c}: CI low": lo, f"{c}: CI high": hi, f"{c}: p": p,
                    f"{c}: informative": bool(kt.mean() <= 1 - MIN_TAPE_LOSS)})
        per_ep += [{"game": g, "pert": c, "episode": e, "agent_kept": a_, "tape_kept": t_} for e, a_, t_ in zip(d.index, ka, kt)]
    rows.append(row)
CLI = pd.DataFrame(rows)
if CLI.empty:
    raise SystemExit("no game had an agent clearly better than random: nothing to score")
for c in ["sticky", "noop"]:
    CLI[f"{c}: p (Holm)"] = holm(CLI[f"{c}: p"])
CLI = CLI.sort_values("sticky: CLI").reset_index(drop=True)
pd.DataFrame(per_ep).to_csv(DIRS["raw"] / "kept_per_episode.csv", index=False)
save_table(CLI, "cli_per_game", "Closed-loop index per game (kept = share of above-random score; CLI = agent kept − tape kept)")

fig, axes = plt.subplots(1, 2, figsize=(10, 0.42 * len(CLI) + 1.6), sharey=True)
y = np.arange(len(CLI))
for ax, c, title in [(axes[0], "sticky", f"Sticky actions (p = {STICKY_P})"), (axes[1], "noop", f"No-op start (0–{NOOP_MAX} decisions)")]:
    ax.barh(y + 0.2, CLI[f"{c}: tape kept"], 0.38, color=GREY, label="blind replay of its own clean games")
    ax.barh(y - 0.2, CLI[f"{c}: agent kept"], 0.38, color=BLUE, label="agent")
    ax.axvline(1, color="#555", lw=0.8, ls="--"); ax.axvline(0, color="#999", lw=0.6)
    for i, r in CLI.iterrows():
        lab = f"CLI {r[f'{c}: CLI']:+.2f}" if r[f"{c}: informative"] else "replay unharmed: not informative"
        ax.text(max(r[f"{c}: agent kept"], r[f"{c}: tape kept"], 0) + 0.03, i, lab, va="center", fontsize=7,
                color="#222" if r[f"{c}: informative"] else "#888")
    ax.set_title(title, fontsize=10); ax.set_xlabel("share of above-random score kept")
axes[0].set_yticks(y, CLI.Game); axes[0].invert_yaxis()
h, l = axes[0].get_legend_handles_labels()
fig.legend(h[::-1], l[::-1], frameon=False, fontsize=8, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=(0, 0.06, 1, 1)); save_fig(fig, "cli_per_game"); plt.show()

## 7. Is a more deterministic policy less closed-loop?

In [ ]:
INF = CLI[CLI["sticky: informative"]]
if len(INF) >= 4:
    rho, p = stats.spearmanr(INF["Entropy"], INF["sticky: CLI"])
    fig, ax = plt.subplots(figsize=(4.6, 3.6))
    for _, r in CLI.iterrows():
        ok = r["sticky: informative"]
        ax.scatter(r.Entropy, r["sticky: CLI"], color=BLUE if ok else "white", edgecolor=BLUE, s=28, zorder=3)
        ax.annotate(r.Game + ("" if ok else " (not informative)"), (r.Entropy, r["sticky: CLI"]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points", color="#222" if ok else "#888")
    ax.axhline(0, color="#999", lw=0.7)
    ax.set_xlabel("policy entropy during clean play (1 = uniform)"); ax.set_ylabel("closed-loop index (sticky)")
    ax.set_title(f"Spearman ρ = {rho:.2f} (p = {p:.2f}, {len(INF)} informative games)", fontsize=10)
    fig.tight_layout(); save_fig(fig, "entropy_vs_cli"); plt.show()
else:
    rho = p = np.nan; print("fewer than 4 scored games: correlation skipped")

## 8. Summary

In [ ]:
lines = ["# Closed-loop index on published Atari PPO agents", "",
         f"Generated {time.strftime('%Y-%m-%d %H:%M')} · {len(GAMES)} games · {EPISODES} episodes per condition", ""]
for _, r in CLI.iterrows():
    lines.append(f"- **{r.Game}**: sticky CLI {r['sticky: CLI']:+.2f} [{r['sticky: CI low']:+.2f}, {r['sticky: CI high']:+.2f}] "
                 f"(agent kept {r['sticky: agent kept']:.2f}, tape kept {r['sticky: tape kept']:.2f}, Holm p {r['sticky: p (Holm)']:.3f}); "
                 f"no-op CLI {r['noop: CLI']:+.2f}; entropy {r.Entropy:.2f}"
                 + ("" if r["sticky: informative"] else " — sticky not informative (tape unharmed)"))
lines.append(f"\nEntropy vs. sticky CLI across informative games: Spearman ρ = {rho:.2f} (p = {p:.2f})")
for name in ["agents", "determinism", "cli_per_game"]:
    lines += ["", (DIRS["tables"] / f"{name}.md").read_text()]
lines += ["", "## Figures", ""] + [f"![{q.stem}](figures/{q.name})" for q in sorted(DIRS["figures"].glob("*.png"))]
(RESULTS / "summary.md").write_text("\n".join(lines) + "\n")
print("\n".join(lines[:len(CLI) + 6]))